# VQE for molecular hydrogen — in about 30 lines

Ground-state energy of H₂ (STO-3G at 0.735 Å) via a hardware-efficient ansatz optimized directly on the QVM simulator.


In [ ]:
import numpy as np
from scipy.optimize import minimize
from qvm.ir import QuantumCircuit
from qvm.simulator import Simulator
from qvm.observable import Hamiltonian

# H2 Hamiltonian (Pauli decomposition), constant offset separate
OFFSET = -1.052373245772859
H = Hamiltonian.from_dict({
    "ZI": -0.39793742484318045, "IZ": 0.39793742484318045,
    "ZZ": -0.01128010425623538, "XX": 0.18093119978423156,
})

def ansatz(p):
    a, b, c, d = p
    qc = QuantumCircuit(2)
    for gate, wire, angle in [("ry",0,a),("ry",1,b)]: qc.add_operation(gate,[wire],params=[angle])
    qc.add_operation("cx",[0,1])
    qc.add_operation("ry",[1],params=[c])
    qc.add_operation("cx",[0,1])
    qc.add_operation("ry",[0],params=[d])
    return qc

def energy(p):
    return OFFSET + Simulator().expectation_value(ansatz(p), H)

In [ ]:
best = min(
    minimize(energy, s, method="Nelder-Mead",
             options={"maxiter": 400, "fatol": 1e-8}).fun
    for s in np.random.default_rng(42).uniform(-np.pi, np.pi, (5, 4))
)
exact = float(np.linalg.eigvalsh(H.to_matrix(2) + OFFSET*np.eye(4))[0])
print(f"VQE  : {best:.6f} Ha\nexact: {exact:.6f} Ha\nerror: {abs(best-exact):.2e} Ha")